# Lava flow annotation

This notebook checks by eye every SHARAD crop a buried lava flow reflector is looked
for in. Each crop is one track over one tile, drawn from its surface down beside its
cluttergram. A reflector is a second line under the surface that the cluttergram
does not show. Each click writes its verdict to
`data/<version>/analysis/verdicts_lavaflow.json` straight away, the version set in
`configs/version.yaml`.

## Setup

In [ ]:
"""Import the candidate crops, the verdicts so far, and what reviews them."""

from mars_multisensor_dataset.analysis.ground_truth.lavaflow.labels import (
    read_lavaflow_labels,
    read_lavaflow_verdicts,
)
from mars_multisensor_dataset.visualization.annotation_lavaflow import review

labels = read_lavaflow_labels()
verdicts = read_lavaflow_verdicts()
crops = {label.crop: label for label in labels}

## What you see

Each crop is one SHARAD track over one 32 km tile, drawn as two panels on the same axes.

- Left, the radargram. Traces run left to right along the track, delay rows run down, and colour is echo power in dB. Only the 256 rows under the atmosphere are drawn. One row is 37.5 ns, about 1.9 m of rock if its permittivity is 9. Samples the noise gate removed are blank.
- Right, the cluttergram. It is the echo a simulation predicts from the surface topography alone, MOLA heights with no subsurface. Whatever it shows comes from the surface, off to the side of the track.
- The title gives the track, the tile and the tile's latitude. The note above the panels says whether the paper names this track, the crop's current verdict, and the region's counts.

The bright line near the top of both panels is the surface echo. Everything is judged under it.

## How to choose

Look under the surface line on the left, then look at the same traces and rows on the right.

- Reflector: a second line under the surface that the cluttergram does not show there. It runs along many traces, lies flat or dips gently, and often rises into the surface at one end, where the flow thins out. The examples below show two.
- None: nothing under the surface, or every line under it has a twin at the same traces and rows in the cluttergram. Also none: a glow in the few rows right under the surface that follows it everywhere, which is the echo's own sidelobe.
- Unsure: a line too faint or too broken to follow, a cluttergram that matches only part of it, or a crop that is mostly blank.

Judge each crop alone. A track with a reflector on one tile can have none on the next, and a track the paper names is not a reflector until you see it. A short line counts as a reflector only if you can follow it; how short is too short is yours to set, and unsure is the place for doubt, since only reflector and none enter the evaluation.

## Stepping through

Pick a region, then use the arrows or click a verdict, which saves it and moves to the next crop. Clicking again on a reviewed crop replaces its verdict. A crop that reads "not on disk" has not been built or downloaded yet.


## Examples

All four are crops of track 5095_01 (`s_00509501`), which Meng et al. 2020 show in their Fig. 2b. Meng et al. sort what they see into three patterns: a single reflector, a reflector that gradually extends to the surface, and both together. The first two are below. The papers define no crop for none or unsure, only the test against the cluttergram, so those two examples are my own reading of that test.

### Reflector that extends to the surface

Left panel, traces 0 to 22: a second line about 16 rows under the surface. It rises to the right and merges with the surface near trace 22. The cluttergram is empty at those traces and rows. This is the pattern of Meng's Fig. 2b.


In [ ]:
"""Draw a reflector that rises into the surface."""

review.crop_panel(crops["b223_c0188/s_00509501"])

### Single reflector

Left panel, traces 38 to 69: a second line about 18 rows under the surface, parallel to it. The cluttergram shows nothing there; its dark shape lower down, from row 2700, has no match on the left.


In [ ]:
"""Draw a single reflector parallel to the surface."""

review.crop_panel(crops["b224_c0187/s_00509501"])

### None

Under the surface there is a glow that fades with depth along the whole crop, and no line can be followed in it.


In [ ]:
"""Draw a crop with no reflector."""

review.crop_panel(crops["b225_c0186/s_00509501"])

### Unsure

Left panel, traces 15 to 27: a faint patch about 15 rows under the surface. The cluttergram is empty there, but the patch is about 12 traces long and too weak to follow as a line.


In [ ]:
"""Draw a crop whose line is too short and faint to call."""

review.crop_panel(crops["b221_c0190/s_00509501"])

## Review

Mark each crop as reflector, none or unsure. A region lists the tracks its paper
names first, each from south to north, then the tracks no paper names. A region
opens on its first crop not yet reviewed.

In [ ]:
"""Judge every candidate crop of a region, track by track."""

review.plot(labels, verdicts)

## Where the review stands

Run the cell again after reviewing. Run `scripts/build_lavaflow.py` again to write
the verdicts into the build's `lavaflow_labels.parquet`.

In [ ]:
"""Tabulate every region's verdicts so far."""

review.table(labels, verdicts)